# 2.3 · Pastas — piezometría PZ0267014

**Tiempo estimado:** 1 h 15 min.

**Objetivos.**

1. Modelar la cota piezométrica `PZ0267014` (Renedo de Esgueva, Valladolid) con un `Pastas.StressModel` lluvia → nivel.
2. Interpretar la función de respuesta a impulsos $\theta$ y el tiempo característico.
3. Añadir un `StressModel` ficticio de **extracciones** para entender la deconvolución de causas.
4. Discutir limitaciones: linealidad, estacionariedad de la respuesta, ausencia de evapotranspiración real.

## Mini-intro Pastas (15 min)

**¿Por qué no usar ARIMA aquí?** Las mediciones de `PZ0267014` son **irregulares** (mensuales o trimestrales según la época, con huecos de años). Box-Jenkins asume frecuencia regular. Pastas, en cambio, modela la cota piezométrica como **convolución** de estímulos (lluvia, bombeo, evapotranspiración) con funciones de respuesta paramétricas:

$$
h(t) = h_0 + \sum_{\tau \le t} R(\tau) \cdot \theta(t - \tau) + \varepsilon(t)
$$

Los **stress series** (lluvia diaria, p. ej.) son regulares; las **observations** (cota piezométrica) pueden ser irregulares. Pastas calibra los parámetros de $\theta$ (forma, escala, tiempo de retardo) por mínimos cuadrados ponderados.

Tres bloques de respuesta clásicos:

- `Gamma` — para recarga lluvia → nivel (forma de gamma asimétrica).
- `Exponential` — para bombeo (decay exponencial).
- `Hantush` — clásica analítica de pozo en acuífero confinado.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pastas as ps

from cst import datos as ud

plt.rcParams.update({"figure.figsize": (10, 3.4), "axes.grid": True, "grid.alpha": 0.3})
print(f"pastas {ps.__version__}")

## 1 · Datos

Cota piezométrica (165 mediciones irregulares 1985-2024) + lluvia diaria ERA5 sobre Valladolid (Open-Meteo).

In [ ]:
piezo = ud.cargar_piezometria("PZ0267014")
lluvia = ud.cargar_lluvia_duero_diaria(fecha_inicio="1985-01-01", fecha_fin="2024-12-31")
lluvia.name = "lluvia_PZ0267014"  # Pastas rechaza nombres con '.'

fig, axes = plt.subplots(2, 1, figsize=(10, 5), sharex=True)
axes[0].plot(piezo.index, piezo.values, marker="o", ms=3, lw=0.6, color="#0d9488")
axes[0].set_ylabel("Cota (m s.n.m.)")
axes[1].plot(lluvia.index, lluvia.values, color="#2563eb", lw=0.3)
axes[1].set_ylabel("Lluvia diaria (mm)")
plt.tight_layout()

## 2 · Modelo Pastas mínimo: cota = f(lluvia)

Un solo *stress* (lluvia) con respuesta tipo `Gamma`.

In [ ]:
ml = ps.Model(piezo, name="PZ0267014")
sm = ps.StressModel(lluvia, rfunc=ps.Gamma(), name="recarga", settings="prec")
ml.add_stressmodel(sm)
ml.solve(report=False, tmin="1985-01-01", tmax="2024-12-31")

print(f"NSE = {ml.stats.nse():.3f}")
print(f"R²  = {ml.stats.rsq():.3f}")
print()
print(ml.parameters[["initial", "optimal", "stderr"]].round(3))

In [ ]:
ml.plot(figsize=(10, 3.5))
plt.tight_layout()

## 3 · La función de respuesta

El parámetro clave: $\theta(t)$ — cuánto sube el nivel piezométrico por un milímetro de lluvia que cae hoy, en función del tiempo transcurrido.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 3.4))
step = ml.get_step_response("recarga")
block = ml.get_block_response("recarga")
ax.plot(block.index, block.values, color="#16a34a", lw=2, label="respuesta a impulso unitario")
ax.plot(step.index, step.values, color="#c2410c", lw=2, ls="--", label="respuesta acumulada (step)")
ax.set_xlabel("Días desde el impulso de lluvia")
ax.set_ylabel("Respuesta (m / mm)")
ax.legend()
plt.tight_layout()

# Tiempo característico (95% acumulado)
t95 = step.index[(step.values / step.values.max()) >= 0.95][0]
print(f"Tiempo característico de respuesta: {t95} días (~{t95 / 365:.1f} años)")

## 4 · Componentes y diagnóstico

Pastas separa el nivel observado en `simulación + residuos + contribución de cada stress`.

In [ ]:
ml.plots.results(figsize=(11, 8))
plt.tight_layout()

## 5 · Añadir un stress de extracciones ficticio

El pozo está en una zona de regadío del Esgueva. Sospechamos extracción veraniega regular. **No tenemos** datos reales de bombeo, así que construimos un stress sintético con forma realista: ~100 m³/día durante los meses de junio-septiembre, modulado por el año hidrológico.

*El objetivo aquí es metodológico — ver cómo Pastas separa contribuciones. La magnitud absoluta no es válida.*

In [ ]:
fechas = lluvia.index
extracciones = pd.Series(
    np.where(fechas.month.isin([6, 7, 8, 9]), 100.0, 0.0),
    index=fechas,
    name="extracciones_ficticias",
)
# Suaviza un poco para que no sea escalón puro
extracciones = extracciones.rolling(15, center=True, min_periods=1).mean()

ml2 = ps.Model(piezo, name="PZ0267014_2stress")
ml2.add_stressmodel(ps.StressModel(lluvia, rfunc=ps.Gamma(), name="recarga", settings="prec"))
ml2.add_stressmodel(
    ps.StressModel(
        extracciones, rfunc=ps.Exponential(), name="extracciones", settings="well", up=False
    )
)  # `up=False` => baja el nivel
ml2.solve(report=False)

print(f"NSE (sólo lluvia)        = {ml.stats.nse():.3f}")
print(f"NSE (lluvia + bombeo)    = {ml2.stats.nse():.3f}")
ml2.plots.results(figsize=(11, 9))
plt.tight_layout()

## 6 · Limitaciones a discutir

1. **Linealidad.** Pastas asume superposición lineal de respuestas. Eventos extremos pueden saturar el suelo o el acuífero → no-linealidad real.
2. **Estacionariedad de la respuesta.** $\theta$ es fija. Si la geometría del acuífero cambia (nuevos pozos, recarga artificial) el modelo no lo captura.
3. **Lluvia no es recarga.** La parte de lluvia que se evapora, escurre o se intercepta no llega al acuífero. Modelos `RechargeModel` con `Berendrecht` o `Linear` incluyen evapotranspiración como segunda input (lo veremos como ejercicio).
4. **Datos sintéticos.** Las extracciones son ficticias — el resultado del modelo de 2 stresses **no** debe interpretarse cuantitativamente.

## 7 · Ejercicios

1. **Otra rfunc.** Reajusta usando `ps.Exponential()` y `ps.Hantush()`. ¿Cuál da mejor NSE? ¿Las formas de respuesta son creíbles?
2. **RechargeModel con evapotranspiración.** Open-Meteo tiene `et0_fao_evapotranspiration` — descarga la ET₀ y monta un `ps.RechargeModel(prec, evap, rfunc=ps.Exponential(), recharge=ps.rch.Linear())`.
3. **Otro pozo.** Hay 654 pozos en la red CHD. Elige otro de la cuenca y ajusta. ¿La respuesta es similar?
4. **Calibración por trozos.** Ajusta el modelo a 1985-2010 (`tmin/tmax`) y predice 2010-2024. Compara con el ajuste global. ¿Hay deriva?
5. **Reto.** Saca la respuesta a impulso unitario "a mano" sin Pastas: convoluciona la lluvia diaria con una Gamma($n, a$) parametrizada y ajusta por mínimos cuadrados a las observaciones. Compara con `ml.get_block_response`.